# Preprocess BridgeData V2 to Masked-HWM Tokens (Kaggle)

This notebook is the BridgeData V2 counterpart to `Novel_dataset_preparation.ipynb` for DROID. It keeps Bridge in a separate run path so the DROID notebook stays clean.

Pipeline:

1. discover BridgeData V2 episodes or local videos/frame folders;
2. load RGB frames from one consistent camera stream;
3. resize/center-crop frames to the Cosmos tokenizer input size;
4. encode non-overlapping 17-frame clips with Cosmos `DV8x8x8`;
5. write the same train/eval folder contract consumed by `RawTokenDataset`;
6. compare the generated Bridge output against an existing standard 1X-style input dataset.

Action conditioning remains disabled for Bridge fine-tuning/evaluation:

```python
WITH_ACT = False
```


In [ ]:
# Settings: Bridge source, tokenizer, and output layout.
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

DATASET_NAME = "bridge_v2"

# One knob for source selection: "gcs", "local", "tfds", or "plain".
BRIDGE_SOURCE_MODE = "gcs"
BRIDGE_GCS_BUILDER_DIR = "gs://gresearch/robotics/bridge/0.1.0/"
BRIDGE_LOCAL_ROOT = Path("/kaggle/input/bridgedata-v2")
BRIDGE_SCAN_ALL_KAGGLE_INPUTS = False
BRIDGE_TFDS_NAME = None
BRIDGE_TFDS_DATA_DIR = KAGGLE_WORKING_ROOT / "raw_datasets" / "tfds"
BRIDGE_ALLOW_TFDS_DOWNLOAD = False
BRIDGE_PLAIN_ROOTS = []

COSMOS_INPUT_ROOT = Path("/kaggle/input/hw-model-2")

OUTPUT_ROOT = KAGGLE_WORKING_ROOT / "bridge_hwm_tokenized_dataset"
TRAIN_DIR = OUTPUT_ROOT / "train_v2.0"
VAL_DIR = OUTPUT_ROOT / "val_v2.0"
FORCE_REWRITE_OUTPUT = True

WINDOW_SIZE = 17
TARGET_SIZE = 256
FRAME_STRIDE = 1
VAL_RATIO = 0.05
MIN_VAL_SOURCES = 4
RANDOM_SEED = 5
DATASET_HZ = 10

# Runtime/data presets. Start with smoke; scale only after validation passes.
# smoke: a few minutes, enough to prove format + finetuning path.
# small: useful first Kaggle dataset, usually well below the 12h session limit.
# medium: larger subset; use only after smoke/small are confirmed.
# full: no artificial cap, not recommended in a single Kaggle session.
RUN_PRESET = "smoke"  # "smoke", "small", "medium", or "full"
PRESET_LIMITS = {
    "smoke":  {"max_episodes": 24,  "max_clips_total": 128,  "max_clips_per_source": 8},
    "small":  {"max_episodes": 120, "max_clips_total": 1000, "max_clips_per_source": 12},
    "medium": {"max_episodes": 500, "max_clips_total": 5000, "max_clips_per_source": 16},
    "full":   {"max_episodes": None, "max_clips_total": None, "max_clips_per_source": None},
}
if RUN_PRESET not in PRESET_LIMITS:
    raise ValueError(f"Unknown RUN_PRESET: {RUN_PRESET}")
MAX_EPISODES = PRESET_LIMITS[RUN_PRESET]["max_episodes"]
MAX_CLIPS_TOTAL = PRESET_LIMITS[RUN_PRESET]["max_clips_total"]
MAX_CLIPS_PER_SOURCE = PRESET_LIMITS[RUN_PRESET]["max_clips_per_source"]

# GCS streaming is much faster when reading early sequential episodes instead of random high indices.
SHUFFLE_SOURCES = False

BATCH_CLIPS = 2
SHARD_SIZE_CLIPS = 1024

CAMERA_KEY_CANDIDATES = [
    "observation.image_0",
    "observation.image_1",
    "observation.image_2",
    "observation.image",
    "observation.images0",
    "observation.images1",
    "observation.images2",
    "observation.wrist_image",
    "observation.wrist_image_left",
    "observation.exterior_image_1_left",
    "image_0",
    "image_1",
    "image_2",
    "image",
    "images0",
    "images1",
    "images2",
]

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

print("Bridge source mode:", BRIDGE_SOURCE_MODE)
print("Bridge GCS builder:", BRIDGE_GCS_BUILDER_DIR)
print("Bridge local root:", BRIDGE_LOCAL_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Preset:", RUN_PRESET)
print("Limits:", {"episodes": MAX_EPISODES, "clips_total": MAX_CLIPS_TOTAL, "clips_per_source": MAX_CLIPS_PER_SOURCE})


In [ ]:
# Runtime dependencies and Cosmos tokenizer discovery.
import importlib
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


def run(cmd, cwd=None, check=True):
    print("\n$", " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


def ensure_import(package_name, pip_name=None):
    try:
        return importlib.import_module(package_name)
    except ModuleNotFoundError:
        run([sys.executable, "-m", "pip", "install", "-q", pip_name or package_name])
        importlib.invalidate_caches()
        return importlib.import_module(package_name)


tf = ensure_import("tensorflow", "tensorflow")
tfds = ensure_import("tensorflow_datasets", "tensorflow_datasets")

try:
    tfds.builder("bridge", data_dir=str(BRIDGE_TFDS_DATA_DIR))
except Exception:
    print("Upgrading TensorFlow Datasets for newer robotics builders...")
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "tensorflow-datasets"])
    importlib.invalidate_caches()
    tfds = importlib.import_module("tensorflow_datasets")

COSMOS_RUNTIME_DEPS = [
    ("mediapy", "mediapy"),
    ("einops", "einops"),
    ("omegaconf", "omegaconf"),
    ("loguru", "loguru"),
    ("safetensors", "safetensors"),
    ("huggingface_hub", "huggingface_hub"),
    ("imageio", "imageio"),
    ("cv2", "opencv-python-headless"),
]
for import_name, pip_name in COSMOS_RUNTIME_DEPS:
    ensure_import(import_name, pip_name)


def import_cosmos_tokenizer_from_source(source_root):
    source_root = Path(source_root)
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))
    importlib.invalidate_caches()
    try:
        from cosmos_tokenizer.video_lib import CausalVideoTokenizer
        return CausalVideoTokenizer
    except ModuleNotFoundError as exc:
        missing = exc.name
        pip_name = {"cv2": "opencv-python-headless", "PIL": "pillow"}.get(missing, missing)
        print(f"Missing dependency while importing Cosmos-Tokenizer: {missing}. Installing {pip_name} and retrying...")
        run([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        from cosmos_tokenizer.video_lib import CausalVideoTokenizer
        return CausalVideoTokenizer


def find_cosmos_tokenizer_source_roots(*roots):
    source_roots = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for package_file in root.rglob("cosmos_tokenizer/video_lib.py"):
            source_roots.append(package_file.parents[1])
    return source_roots


def ensure_cosmos_tokenizer_package():
    try:
        from cosmos_tokenizer.video_lib import CausalVideoTokenizer
        return CausalVideoTokenizer
    except ModuleNotFoundError:
        pass

    for source_root in find_cosmos_tokenizer_source_roots(COSMOS_INPUT_ROOT, KAGGLE_INPUT_ROOT, KAGGLE_WORKING_ROOT):
        try:
            print("Using Cosmos-Tokenizer source from:", source_root)
            return import_cosmos_tokenizer_from_source(source_root)
        except ModuleNotFoundError:
            continue

    cosmos_repo = KAGGLE_WORKING_ROOT / "Cosmos-Tokenizer"
    if not cosmos_repo.exists():
        print("Cosmos-Tokenizer source not found; cloning NVIDIA/Cosmos-Tokenizer...")
        run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/Cosmos-Tokenizer.git", str(cosmos_repo)])

    for source_root in find_cosmos_tokenizer_source_roots(cosmos_repo):
        print("Using Cosmos-Tokenizer source from:", source_root)
        return import_cosmos_tokenizer_from_source(source_root)

    run([sys.executable, "-m", "pip", "install", "-e", str(cosmos_repo)])
    importlib.invalidate_caches()
    from cosmos_tokenizer.video_lib import CausalVideoTokenizer
    return CausalVideoTokenizer


def find_first_file(filename, roots):
    candidates = []
    for root in roots:
        root = Path(root)
        if root.exists():
            direct = root / filename
            if direct.exists():
                candidates.append(direct)
            candidates.extend(root.rglob(filename))
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        return candidate
    return None


CausalVideoTokenizer = ensure_cosmos_tokenizer_package()
ENCODER_JIT = find_first_file("encoder.jit", [COSMOS_INPUT_ROOT, KAGGLE_INPUT_ROOT, Path.cwd()])
DECODER_JIT = find_first_file("decoder.jit", [COSMOS_INPUT_ROOT, KAGGLE_INPUT_ROOT, Path.cwd()])

if ENCODER_JIT is None:
    raise FileNotFoundError("Could not find encoder.jit. Add Cosmos-0.1-Tokenizer-DV8x8x8 as a Kaggle input.")
print("tensorflow_datasets:", tfds.__version__)
print("Encoder:", ENCODER_JIT)
print("Decoder:", DECODER_JIT)


In [ ]:
# BridgeData V2 source discovery.
import random
from pathlib import Path


def is_gcs_path(path):
    return isinstance(path, str) and path.startswith("gs://")


def find_tfds_builder_dirs(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p.parent for p in root.rglob("dataset_info.json"))


def get_split_names(builder):
    names = list(builder.info.splits.keys())
    if "train" in names:
        return ["train"]
    return names[:1]


def local_bridge_builder_dirs():
    roots = []
    search_roots = [BRIDGE_LOCAL_ROOT]
    if BRIDGE_SCAN_ALL_KAGGLE_INPUTS:
        search_roots.append(KAGGLE_INPUT_ROOT)
    for root in search_roots:
        roots.extend(find_tfds_builder_dirs(root))
    return roots


def tfds_named_builder_dirs():
    if not BRIDGE_TFDS_NAME:
        return []
    builder = tfds.builder(BRIDGE_TFDS_NAME, data_dir=str(BRIDGE_TFDS_DATA_DIR))
    builder_dir = Path(builder.data_dir)
    print("Bridge TFDS builder:", builder.name, builder.version, "dir:", builder_dir)
    if not builder_dir.exists():
        if BRIDGE_ALLOW_TFDS_DOWNLOAD:
            print("Downloading/preparing Bridge TFDS. This can be hundreds of GiB:", builder_dir)
            builder.download_and_prepare()
        else:
            print("Bridge TFDS is not downloaded. Keep BRIDGE_ALLOW_TFDS_DOWNLOAD=False unless disk is sufficient.")
            return []
    return [builder_dir]


def bridge_builder_dirs():
    mode = BRIDGE_SOURCE_MODE.lower()
    roots = []
    if mode == "gcs":
        roots.append(BRIDGE_GCS_BUILDER_DIR)
    elif mode == "local":
        roots.extend(local_bridge_builder_dirs())
    elif mode == "tfds":
        roots.extend(tfds_named_builder_dirs())
    elif mode == "plain":
        return []
    else:
        raise ValueError(f"Unknown BRIDGE_SOURCE_MODE: {BRIDGE_SOURCE_MODE}")

    seen = set()
    unique = []
    for root in roots:
        if is_gcs_path(root):
            key = root.rstrip("/")
            value = root
        else:
            root_path = Path(root)
            key = str(root_path.resolve()) if root_path.exists() else str(root_path)
            value = root_path
        if key not in seen:
            seen.add(key)
            unique.append(value)
    return unique


def discover_rlds_sources(builder_dirs):
    sources = []
    for builder_dir in builder_dirs:
        try:
            builder = tfds.builder_from_directory(str(builder_dir))
            split_names = get_split_names(builder)
            print("Bridge builder:", builder.name, builder.version, "splits:", split_names, "dir:", builder_dir)
            for split in split_names:
                n = builder.info.splits[split].num_examples
                for episode_index in range(n):
                    sources.append(("rlds", {"builder_dir": str(builder_dir), "split": split, "episode_index": episode_index}))
        except Exception as exc:
            print("Skipping invalid TFDS builder dir:", builder_dir, repr(exc))
    return sources


def find_video_files(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)


def find_frame_dirs(root):
    root = Path(root)
    if not root.exists():
        return []
    frame_dirs = []
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        imgs = [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if len(imgs) >= WINDOW_SIZE:
            frame_dirs.append(d)
    return sorted(frame_dirs)


def discover_plain_sources():
    roots = BRIDGE_PLAIN_ROOTS or [BRIDGE_LOCAL_ROOT]
    sources = []
    for root in roots:
        sources.extend(("video", p) for p in find_video_files(root))
        sources.extend(("frames", p) for p in find_frame_dirs(root))
    return sources


def discover_bridge_sources():
    builder_dirs = bridge_builder_dirs()
    print("Bridge TFDS builder dirs:")
    for d in builder_dirs[:20]:
        print("-", d)

    sources = discover_rlds_sources(builder_dirs)
    if not sources:
        print("No Bridge RLDS sources found; scanning videos/frame folders as fallback.")
        sources = discover_plain_sources()

    if SHUFFLE_SOURCES:
        random.Random(RANDOM_SEED).shuffle(sources)
    if MAX_EPISODES is not None:
        sources = sources[:MAX_EPISODES]

    print("Bridge sources:", len(sources))
    for item in sources[:10]:
        print(item)
    if not sources:
        raise RuntimeError("No Bridge sources found. Check BRIDGE_SOURCE_MODE and the source paths in the settings cell.")
    return sources


sources = discover_bridge_sources()


In [ ]:
# Frame loading and preprocessing helpers.
import cv2
import numpy as np
import torch
from PIL import Image
from collections.abc import Mapping

_BUILDER_CACHE = {}
_DATASET_CACHE = {}
_EPISODE_FRAME_CACHE = {}
_PRINTED_CAMERA_KEYS = set()


def resize_center_crop_rgb(frame_rgb, target_size=TARGET_SIZE):
    img = Image.fromarray(frame_rgb.astype(np.uint8)).convert("RGB")
    w, h = img.size
    scale = target_size / min(w, h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    img = img.resize((new_w, new_h), Image.BICUBIC)
    left = (new_w - target_size) // 2
    top = (new_h - target_size) // 2
    img = img.crop((left, top, left + target_size, top + target_size))
    return np.asarray(img, dtype=np.uint8)


def get_nested_value(node, dotted_key):
    cur = node
    for part in dotted_key.split("."):
        if not isinstance(cur, Mapping) or part not in cur:
            return None
        cur = cur[part]
    return cur


def normalize_frame_array(frame):
    frame = np.asarray(frame)
    if frame.ndim == 4 and frame.shape[0] == 1:
        frame = frame[0]
    if frame.ndim == 2:
        frame = np.repeat(frame[..., None], 3, axis=-1)
    if frame.ndim != 3:
        return None
    if frame.shape[0] in (1, 3) and frame.shape[-1] not in (1, 3):
        frame = np.moveaxis(frame, 0, -1)
    if frame.shape[-1] == 1:
        frame = np.repeat(frame, 3, axis=-1)
    if frame.shape[-1] != 3:
        return None
    if frame.dtype != np.uint8:
        if frame.max(initial=0) <= 1.0:
            frame = frame * 255.0
        frame = np.clip(frame, 0, 255).astype(np.uint8)
    return frame


def find_first_image_in_mapping(node, prefix=""):
    if isinstance(node, Mapping):
        for k, v in node.items():
            child_prefix = f"{prefix}.{k}" if prefix else str(k)
            found = find_first_image_in_mapping(v, child_prefix)
            if found is not None:
                return found
        return None
    frame = normalize_frame_array(node)
    if frame is not None:
        return frame, prefix
    return None


def pick_step_frame(step):
    for key in CAMERA_KEY_CANDIDATES:
        frame = get_nested_value(step, key)
        frame = normalize_frame_array(frame) if frame is not None else None
        if frame is not None:
            return frame, key

    found = find_first_image_in_mapping(step)
    if found is not None:
        return found
    raise KeyError(f"Could not find an image camera in a Bridge step. Tried: {CAMERA_KEY_CANDIDATES}")


def get_builder(builder_dir):
    if builder_dir not in _BUILDER_CACHE:
        _BUILDER_CACHE[builder_dir] = tfds.builder_from_directory(str(builder_dir))
    return _BUILDER_CACHE[builder_dir]


def load_single_rlds_episode(builder_dir, split, episode_index):
    # Direct TFDS absolute slicing avoids O(N^2) scans where each episode load starts from the beginning.
    builder = get_builder(builder_dir)
    split_slice = tfds.core.ReadInstruction(split, from_=int(episode_index), to=int(episode_index) + 1, unit="abs")
    dataset = tfds.as_numpy(builder.as_dataset(split=split_slice, shuffle_files=False))
    try:
        return next(iter(dataset))
    except StopIteration as exc:
        raise IndexError(f"Episode index out of range: {episode_index}") from exc


def load_rlds_episode_frames(source_info):
    builder_dir = source_info["builder_dir"]
    split = source_info["split"]
    episode_index = source_info["episode_index"]
    cache_key = (builder_dir, split, episode_index, FRAME_STRIDE, TARGET_SIZE)
    if cache_key in _EPISODE_FRAME_CACHE:
        return _EPISODE_FRAME_CACHE[cache_key]

    episode = load_single_rlds_episode(builder_dir, split, episode_index)
    if "steps" not in episode:
        raise KeyError(f"Episode has no 'steps'. Top-level keys: {list(episode.keys())}")

    frames = []
    camera_key = None
    for step_i, step in enumerate(episode["steps"]):
        if step_i % FRAME_STRIDE != 0:
            continue
        frame, used_key = pick_step_frame(step)
        camera_key = camera_key or used_key
        frames.append(resize_center_crop_rgb(frame))

    if not frames:
        raise RuntimeError(f"Episode {episode_index} has no usable frames after FRAME_STRIDE={FRAME_STRIDE}")

    print_once_key = (builder_dir, split, episode_index)
    if print_once_key not in _PRINTED_CAMERA_KEYS:
        print(f"Episode {episode_index} camera: {camera_key}, frames: {len(frames)}")
        _PRINTED_CAMERA_KEYS.add(print_once_key)

    _EPISODE_FRAME_CACHE[cache_key] = frames
    return frames


def load_video_frames(path):
    cap = cv2.VideoCapture(str(path))
    frames = []
    i = 0
    ok, frame_bgr = cap.read()
    while ok:
        if i % FRAME_STRIDE == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append(resize_center_crop_rgb(frame_rgb))
        i += 1
        ok, frame_bgr = cap.read()
    cap.release()
    return frames


def load_frame_dir_frames(path):
    image_paths = sorted(p for p in Path(path).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    frames = []
    for i, image_path in enumerate(image_paths):
        if i % FRAME_STRIDE != 0:
            continue
        frame_rgb = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.uint8)
        frames.append(resize_center_crop_rgb(frame_rgb))
    return frames


def load_source_frames(kind, payload):
    if kind == "rlds":
        return load_rlds_episode_frames(payload)
    if kind == "video":
        return load_video_frames(payload)
    if kind == "frames":
        return load_frame_dir_frames(payload)
    raise ValueError(f"Unknown source kind: {kind}")


def iter_nonoverlap_windows(frames):
    usable = (len(frames) // WINDOW_SIZE) * WINDOW_SIZE
    for start in range(0, usable, WINDOW_SIZE):
        yield np.stack(frames[start:start + WINDOW_SIZE], axis=0)


def clips_to_cosmos_tensor(clips):
    arr = np.stack(clips, axis=0)
    x = torch.from_numpy(arr).permute(0, 4, 1, 2, 3).float()
    x = x / 127.5 - 1.0
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32
    return x.to(device=device, dtype=dtype)


def inspect_first_source():
    kind, payload = sources[0]
    print("Inspecting first Bridge source:", kind, payload)
    frames = load_source_frames(kind, payload)
    print("Frames loaded:", len(frames))
    if frames:
        print("First frame:", frames[0].shape, frames[0].dtype, int(frames[0].min()), int(frames[0].max()))
        print("Non-overlap clips available:", sum(1 for _ in iter_nonoverlap_windows(frames)))


inspect_first_source()


In [ ]:
# Convert BridgeData V2 sources to Masked-HWM tokenized train/val layout.
import json
import math
import shutil
from tqdm.auto import tqdm

encoder = CausalVideoTokenizer(checkpoint_enc=str(ENCODER_JIT))


def prepare_output_dirs():
    if OUTPUT_ROOT.exists():
        if FORCE_REWRITE_OUTPUT:
            shutil.rmtree(OUTPUT_ROOT)
        else:
            raise FileExistsError(f"Output already exists: {OUTPUT_ROOT}. Set FORCE_REWRITE_OUTPUT=True to replace it.")
    (TRAIN_DIR / "metadata").mkdir(parents=True, exist_ok=True)
    (TRAIN_DIR / "videos").mkdir(parents=True, exist_ok=True)
    (TRAIN_DIR / "segment_indices").mkdir(parents=True, exist_ok=True)
    VAL_DIR.mkdir(parents=True, exist_ok=True)


def write_split(split_name, split_sources, output_dir, is_val=False):
    shard_id = 0
    shard_tokens = []
    shard_segment_ids = []
    shard_frames = 0
    segment_id = 0
    total_clips = 0
    total_frames = 0
    shard_metas = []

    def flush_shard():
        nonlocal shard_id, shard_tokens, shard_segment_ids, shard_frames
        if not shard_tokens:
            return
        tokens_arr = np.concatenate(shard_tokens, axis=0).astype(np.int32)
        segments_arr = np.asarray(shard_segment_ids, dtype=np.int32)

        if is_val:
            tokens_arr.tofile(output_dir / f"video_{shard_id}.bin")
            segments_arr.tofile(output_dir / f"segment_idx_{shard_id}.bin")
            (output_dir / f"metadata_{shard_id}.json").write_text(json.dumps({"shard_num_frames": shard_frames}))
        else:
            tokens_arr.tofile(output_dir / "videos" / f"video_{shard_id}.bin")
            segments_arr.tofile(output_dir / "segment_indices" / f"segment_idx_{shard_id}.bin")
            (output_dir / "metadata" / f"metadata_{shard_id}.json").write_text(json.dumps({"shard_num_frames": shard_frames}))

        shard_metas.append({"shard": shard_id, "clips": int(tokens_arr.shape[0]), "frames": int(shard_frames)})
        print(f"Wrote {split_name} shard {shard_id}: clips={tokens_arr.shape[0]}, frames={shard_frames}")
        shard_id += 1
        shard_tokens = []
        shard_segment_ids = []
        shard_frames = 0

    for kind, payload in tqdm(split_sources, desc=f"Encoding {split_name}"):
        if MAX_CLIPS_TOTAL is not None and total_clips >= MAX_CLIPS_TOTAL:
            break

        try:
            frames = load_source_frames(kind, payload)
        except Exception as exc:
            print("Skipping unreadable source:", kind, payload, repr(exc))
            continue

        windows = list(iter_nonoverlap_windows(frames))
        if MAX_CLIPS_PER_SOURCE is not None:
            windows = windows[:MAX_CLIPS_PER_SOURCE]
        if len(windows) < 2:
            continue

        for batch_start in range(0, len(windows), BATCH_CLIPS):
            if MAX_CLIPS_TOTAL is not None and total_clips >= MAX_CLIPS_TOTAL:
                break
            batch_windows = windows[batch_start:batch_start + BATCH_CLIPS]
            if MAX_CLIPS_TOTAL is not None:
                batch_windows = batch_windows[:MAX_CLIPS_TOTAL - total_clips]
            if not batch_windows:
                break

            x = clips_to_cosmos_tensor(batch_windows)
            with torch.no_grad():
                indices, _ = encoder.encode(x)
            tokens = indices.detach().cpu().numpy().astype(np.int32)
            if tokens.shape[1:] != (3, 32, 32):
                raise ValueError(f"Unexpected token shape {tokens.shape}; expected (B,3,32,32). Check tokenizer variant/input size.")

            shard_tokens.append(tokens)
            frames_in_batch = len(batch_windows) * WINDOW_SIZE
            shard_segment_ids.extend([segment_id] * frames_in_batch)
            shard_frames += frames_in_batch
            total_clips += len(batch_windows)
            total_frames += frames_in_batch

            if sum(t.shape[0] for t in shard_tokens) >= SHARD_SIZE_CLIPS:
                flush_shard()

        segment_id += 1

    flush_shard()
    (output_dir / "metadata.json").write_text(json.dumps({"num_shards": shard_id, "hz": DATASET_HZ}, indent=2))
    (output_dir / "preprocess_summary.json").write_text(json.dumps({
        "dataset": DATASET_NAME,
        "split": split_name,
        "num_sources": len(split_sources),
        "total_clips": total_clips,
        "total_frames": total_frames,
        "window_size": WINDOW_SIZE,
        "frame_stride": FRAME_STRIDE,
        "target_size": TARGET_SIZE,
        "run_preset": RUN_PRESET,
        "max_clips_total": MAX_CLIPS_TOTAL,
        "min_val_sources": MIN_VAL_SOURCES,
        "max_clips_per_source": MAX_CLIPS_PER_SOURCE,
        "token_shape": [3, 32, 32],
        "shards": shard_metas,
    }, indent=2))
    return total_clips, total_frames, shard_id


prepare_output_dirs()
val_count = min(len(sources) - 1, max(MIN_VAL_SOURCES, int(len(sources) * VAL_RATIO))) if len(sources) > 1 else 0
val_sources = sources[:val_count]
train_sources = sources[val_count:]
if not train_sources and val_sources:
    train_sources, val_sources = val_sources, []

print("Bridge train sources:", len(train_sources))
print("Bridge val sources:", len(val_sources))
train_stats = write_split("bridge_train", train_sources, TRAIN_DIR, is_val=False)
val_stats = write_split("bridge_val", val_sources, VAL_DIR, is_val=True) if val_sources else (0, 0, 0)
print("Bridge train stats:", train_stats)
print("Bridge val stats:", val_stats)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# Validate Bridge output files and optionally decode a sample clip.
import json
import math
import numpy as np
import torch

print("Output tree sample:")
for p in sorted(OUTPUT_ROOT.rglob("*"))[:80]:
    print("-", p.relative_to(OUTPUT_ROOT))

train_meta = json.loads((TRAIN_DIR / "metadata.json").read_text())
assert train_meta["num_shards"] > 0, "No train shards were written. Increase MAX_EPISODES or check camera/source settings."
shard_meta = json.loads((TRAIN_DIR / "metadata" / "metadata_0.json").read_text())
num_chunks = math.ceil(shard_meta["shard_num_frames"] / WINDOW_SIZE)
tokens = np.memmap(TRAIN_DIR / "videos" / "video_0.bin", dtype=np.int32, mode="r", shape=(num_chunks, 3, 32, 32))
segments = np.memmap(TRAIN_DIR / "segment_indices" / "segment_idx_0.bin", dtype=np.int32, mode="r", shape=(shard_meta["shard_num_frames"],))
print("First train shard token shape:", tokens.shape)
print("First train shard segment shape:", segments.shape)
print("First token min/max:", int(np.asarray(tokens[0]).min()), int(np.asarray(tokens[0]).max()))

if DECODER_JIT is not None and torch.cuda.is_available():
    decoder = CausalVideoTokenizer(checkpoint_dec=str(DECODER_JIT))
    sample = torch.from_numpy(np.asarray(tokens[0:1]).copy()).long().cuda()
    with torch.no_grad():
        decoded = decoder.decode(sample).detach().float().cpu()
    print("Decoded sample shape:", tuple(decoded.shape))
elif DECODER_JIT is None:
    print("Skipping decode validation because decoder.jit was not found.")
else:
    print("Skipping decode validation because CUDA is not available.")


In [ ]:
# Compare generated Bridge tokens against the standard 1X/world-model input dataset structure.
import json
import math
from pathlib import Path

import numpy as np

REFERENCE_1X_ROOT = None
GENERATED_BRIDGE_ROOT = OUTPUT_ROOT


def status_line(ok, label, detail=""):
    tag = "OK" if ok else "FAIL"
    msg = f"[{tag}] {label}"
    if detail:
        msg += f": {detail}"
    print(msg)
    return ok


def find_dataset_roots(search_root=KAGGLE_INPUT_ROOT):
    roots = []
    search_root = Path(search_root)
    if not search_root.exists():
        return roots
    for meta in search_root.rglob("train_v2.0/metadata.json"):
        root = meta.parents[1]
        if (root / "val_v2.0" / "metadata.json").exists():
            roots.append(root)
    roots = sorted(set(roots), key=lambda x: ("world-model" not in str(x).lower(), len(str(x))))
    return roots


def auto_reference_root():
    if REFERENCE_1X_ROOT is not None:
        return Path(REFERENCE_1X_ROOT)
    roots = find_dataset_roots()
    generated = Path(GENERATED_BRIDGE_ROOT).resolve()
    for root in roots:
        try:
            if root.resolve() == generated:
                continue
        except FileNotFoundError:
            pass
        return root
    raise FileNotFoundError(
        "Could not auto-detect a reference 1X dataset under /kaggle/input. "
        "Set REFERENCE_1X_ROOT manually to the folder containing train_v2.0 and val_v2.0."
    )


def split_paths(root, split):
    root = Path(root)
    split_dir = root / f"{split}_v2.0"
    is_val = split == "val"
    return {
        "split_dir": split_dir,
        "metadata": split_dir / "metadata.json",
        "shard_meta": (split_dir / "metadata_0.json") if is_val else (split_dir / "metadata" / "metadata_0.json"),
        "video": (split_dir / "video_0.bin") if is_val else (split_dir / "videos" / "video_0.bin"),
        "segment": (split_dir / "segment_idx_0.bin") if is_val else (split_dir / "segment_indices" / "segment_idx_0.bin"),
        "robot_states": None if is_val else (split_dir / "robot_states" / "states_0.bin"),
    }


def inspect_split(root, split, window_size=WINDOW_SIZE):
    paths = split_paths(root, split)
    result = {
        "root": str(root),
        "split": split,
        "ok": True,
        "paths": {k: str(v) if v is not None else None for k, v in paths.items()},
    }

    required = ["split_dir", "metadata", "shard_meta", "video", "segment"]
    for key in required:
        exists = paths[key].exists()
        result[f"{key}_exists"] = exists
        result["ok"] &= exists
    if not result["ok"]:
        return result

    meta = json.loads(paths["metadata"].read_text())
    shard_meta = json.loads(paths["shard_meta"].read_text())
    frames = int(shard_meta["shard_num_frames"])
    chunks = math.ceil(frames / window_size)
    video_bytes = paths["video"].stat().st_size
    segment_bytes = paths["segment"].stat().st_size
    tokens_per_chunk = video_bytes // max(1, chunks * np.dtype(np.int32).itemsize)
    token_shape_ok = tokens_per_chunk == 3 * 32 * 32
    video_size_ok = video_bytes == chunks * 3 * 32 * 32 * np.dtype(np.int32).itemsize
    segment_size_ok = segment_bytes == frames * np.dtype(np.int32).itemsize

    result.update({
        "num_shards": int(meta.get("num_shards", -1)),
        "hz": meta.get("hz"),
        "shard_num_frames": frames,
        "chunks": chunks,
        "video_bytes": video_bytes,
        "segment_bytes": segment_bytes,
        "tokens_per_chunk": tokens_per_chunk,
        "token_shape": [3, 32, 32] if token_shape_ok else [tokens_per_chunk],
        "token_shape_ok": token_shape_ok,
        "video_size_ok": video_size_ok,
        "segment_size_ok": segment_size_ok,
        "robot_states_exists": bool(paths["robot_states"] and paths["robot_states"].exists()),
    })
    result["ok"] &= result["num_shards"] > 0 and token_shape_ok and video_size_ok and segment_size_ok

    if result["ok"]:
        tokens = np.memmap(paths["video"], dtype=np.int32, mode="r", shape=(chunks, 3, 32, 32))
        segments = np.memmap(paths["segment"], dtype=np.int32, mode="r", shape=(frames,))
        effective_chunks = chunks - (1 if frames % window_size != 0 else 0)
        valid_examples = 0
        for i in range(max(0, effective_chunks - 1)):
            idx1 = i * window_size
            idx2 = (i + 1) * window_size - 1
            if idx2 < len(segments) and segments[idx1] == segments[idx2]:
                valid_examples += 1
        result.update({
            "dtype": str(tokens.dtype),
            "first_token_min": int(np.asarray(tokens[0]).min()) if chunks else None,
            "first_token_max": int(np.asarray(tokens[0]).max()) if chunks else None,
            "unique_segments_first_64": int(len(np.unique(np.asarray(segments[:min(len(segments), 64)])))),
            "raw_token_dataset_valid_examples_estimate": valid_examples,
        })
    return result


def print_split_report(name, report):
    print(f"\n{name} {report['split']} split")
    status_line(report.get("split_dir_exists", False), "split directory", report["paths"]["split_dir"])
    status_line(report.get("metadata_exists", False), "metadata.json")
    status_line(report.get("shard_meta_exists", False), "first shard metadata")
    status_line(report.get("video_exists", False), "first video bin")
    status_line(report.get("segment_exists", False), "first segment bin")
    if not all(report.get(f"{k}_exists", False) for k in ["split_dir", "metadata", "shard_meta", "video", "segment"]):
        return
    status_line(report["num_shards"] > 0, "num_shards", str(report["num_shards"]))
    status_line(report["token_shape_ok"], "token shape", str(report["token_shape"]))
    status_line(report["video_size_ok"], "video file size matches metadata", f"chunks={report['chunks']}, bytes={report['video_bytes']}")
    status_line(report["segment_size_ok"], "segment file size matches metadata", f"frames={report['shard_num_frames']}, bytes={report['segment_bytes']}")
    print("hz:", report.get("hz"))
    print("robot_states exists:", report.get("robot_states_exists"), "(required only when WITH_ACT=True)")
    print("RawTokenDataset valid examples estimate:", report.get("raw_token_dataset_valid_examples_estimate"))
    print("first token min/max:", report.get("first_token_min"), report.get("first_token_max"))


reference_root = auto_reference_root()
generated_root = Path(GENERATED_BRIDGE_ROOT)
print("Reference 1X root:", reference_root)
print("Generated Bridge root:", generated_root)

reports = {}
for split in ["train", "val"]:
    reports[("1x", split)] = inspect_split(reference_root, split)
    reports[("bridge", split)] = inspect_split(generated_root, split)
    print_split_report("1X reference", reports[("1x", split)])
    print_split_report("Bridge generated", reports[("bridge", split)])

print("\nStructure comparison summary")
for split in ["train", "val"]:
    ref = reports[("1x", split)]
    gen = reports[("bridge", split)]
    comparable = ref["ok"] and gen["ok"]
    same_required_layout = all(ref.get(key) == gen.get(key) for key in ["token_shape_ok", "video_size_ok", "segment_size_ok"])
    status_line(comparable and same_required_layout, f"{split}_v2.0 required layout matches")

print("\nTraining readiness")
train_ok = reports[("bridge", "train")]["ok"] and reports[("bridge", "train")].get("raw_token_dataset_valid_examples_estimate", 0) > 0
val_ok = reports[("bridge", "val")]["ok"] and reports[("bridge", "val")].get("raw_token_dataset_valid_examples_estimate", 0) > 0
status_line(train_ok, "Bridge train split can be read by RawTokenDataset-style loader")
status_line(val_ok, "Bridge val split can be read by RawTokenDataset-style loader")
print("Use WITH_ACT=False for Bridge unless you add a Bridge-to-1X action adapter.")

comparison_payload = {
    "reference_root": str(reference_root),
    "generated_bridge_root": str(generated_root),
    "reports": {f"{dataset}_{split}": report for (dataset, split), report in reports.items()},
    "training_readiness": {"train": bool(train_ok), "val": bool(val_ok), "with_act": False},
}
(OUTPUT_ROOT / "structure_comparison.json").write_text(json.dumps(comparison_payload, indent=2))
print("Saved comparison:", OUTPUT_ROOT / "structure_comparison.json")


In [ ]:
# Optional: publish /kaggle/working/bridge_hwm_tokenized_dataset as a private Kaggle Dataset.
import json
import os
import re
import subprocess
import sys
from pathlib import Path

RUN_KAGGLE_UPLOAD = False
PUBLISH_ROOT = Path(OUTPUT_ROOT)
KAGGLE_DATASET_SLUG = "bridgedata-v2-hwm-tokenized"
KAGGLE_DATASET_TITLE = "BridgeData V2 Masked-HWM Cosmos Tokens"
KAGGLE_DATASET_LICENSE = "CC-BY-4.0"
MAKE_PRIVATE = True


def require_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(
            f"Missing Kaggle secret {name}. Add it under Add-ons -> Secrets and enable it for this notebook."
        ) from exc


if RUN_KAGGLE_UPLOAD:
    if not PUBLISH_ROOT.exists():
        raise FileNotFoundError(f"Tokenized output does not exist: {PUBLISH_ROOT}")
    for required in ["train_v2.0/metadata.json", "val_v2.0/metadata.json"]:
        if not (PUBLISH_ROOT / required).exists():
            raise FileNotFoundError(f"Missing required output file: {PUBLISH_ROOT / required}")

    username = require_secret("KAGGLE_USERNAME").strip()
    api_key = require_secret("KAGGLE_KEY").strip()
    if not re.fullmatch(r"[a-z0-9-]+", KAGGLE_DATASET_SLUG):
        raise ValueError("KAGGLE_DATASET_SLUG must contain only lowercase letters, numbers, and hyphens.")

    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = api_key
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

    metadata = {
        "id": f"{username}/{KAGGLE_DATASET_SLUG}",
        "title": KAGGLE_DATASET_TITLE,
        "licenses": [{"name": KAGGLE_DATASET_LICENSE}],
        "isPrivate": MAKE_PRIVATE,
    }
    (PUBLISH_ROOT / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    cmd = ["kaggle", "datasets", "create", "-p", str(PUBLISH_ROOT), "--dir-mode", "zip"]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        if "already exists" in result.stdout.lower():
            raise RuntimeError(
                "This dataset slug already exists. Change KAGGLE_DATASET_SLUG for a new dataset, "
                "or use `kaggle datasets version -p <folder> -m <message> --dir-mode zip` to update it."
            )
        raise RuntimeError(f"Kaggle dataset creation failed with exit code {result.returncode}.")
    print("Created Kaggle Dataset:", f"https://www.kaggle.com/datasets/{username}/{KAGGLE_DATASET_SLUG}")
else:
    print("Upload skipped. Set RUN_KAGGLE_UPLOAD=True after verification if you want to publish the tokenized Bridge dataset.")


## Outputs

- Tokenized Bridge dataset: `/kaggle/working/bridge_hwm_tokenized_dataset`
- Standard-structure comparison: `/kaggle/working/bridge_hwm_tokenized_dataset/structure_comparison.json`
- Recommended training/evaluation setting: `WITH_ACT=False`

For a smoke run, keep `RUN_PRESET="smoke"`. For a stronger but still Kaggle-friendly subset, use `RUN_PRESET="small"`. Only use `medium`/`full` after the structure comparison passes and you have enough session time.
